# Week 6 - Data Wrangling Before Ingestion Pipelines (Student Version)

In this notebook we will practice **data cleaning, preprocessing, and preparation** using a *messy* version of the UCI Student Performance dataset.

Topics covered:
- Inspecting a dirty dataset
- Fixing separators and accidental index columns
- Standardizing column names
- Understanding categorical vs numerical data
- Handling inconsistent values
- Detecting and imputing missing values
- Fixing column types
- Checking duplicates and outliers
- Preparing clean data before a future ingestion pipeline

## SETUP


In [119]:
# uncomment if you need any of those
# !pip install pandas numpy


**Import necessary libraries**


In [122]:
import pandas as pd
import numpy as np
from pathlib import Path


## Dataset

In this section, we will work with the **dirty** versions of the UCI Student Performance dataset.

Files:
- `../../raw/uci-student-performance/dirty/student-mat-dirty.csv`
- `../../raw/uci-student-performance/dirty/student-por-dirty.csv`

We are not loading clean source files directly in this practical.
The objective is to clean the dirty files first and prepare them for later MongoDB insertion.

**Data dictionary (selected fields)**

| Column | Description |
|---|---|
| school | Student school (`GP` or `MS`) |
| sex | Student sex |
| age | Student age |
| address | Home address type (`U` or `R`) |
| studytime | Weekly study time category |
| failures | Number of past class failures |
| absences | Number of school absences |
| G1 | First period grade |
| G2 | Second period grade |
| G3 | Final grade |
| internet | Internet access at home |
| higher | Intention to pursue higher education |


## 1. Inspecting the dataset

### Q1.1 Define the paths to both dirty files


In [126]:
!dir

 Volume in drive C is Windows
 Volume Serial Number is F8DA-AC7E

 Directory of C:\Users\ineko\prj\src

05/12/2026  03:09 PM    <DIR>          .
05/05/2026  04:41 PM    <DIR>          ..
05/12/2026  02:26 PM    <DIR>          .ipynb_checkpoints
05/12/2026  03:02 PM            34,723 5-pymongo-crud-ingestion-student.ipynb
05/12/2026  03:09 PM            53,414 6-data-wrangling-student.ipynb
               2 File(s)         88,137 bytes
               3 Dir(s)  345,163,468,800 bytes free


In [128]:
# your code here
raw_mat = Path("../raw/uci-student-performance/dirty/student-mat-dirty.csv")
raw_pt = Path("../raw/uci-student-performance/dirty/student-por-dirty.csv")

### Q1.2 Load the Mathematics dataset

Before anything, we must load the dataset correctly.

Manually inspect the dataset file or use commands such as:
- `!head file_name.csv`
- `!type file_name.csv`

Try first without extra parameters and inspect the result.


In [131]:
# your code here
df_mat = pd.read_csv(raw_mat)

In [133]:
df_mat.head()

,;school;sex;age;address;famsize; pstatus ; medu ; fedu ;mjob ;fjob ;reason;guardian;traveltime;studytime;failures;schoolsup;famsup;paid;activities;nursery;higher;internet;romantic;famrel;freetime;goout;Dalc;Walc;health;absences;G1;G2;G3
0,0;GP;F;18;U;GT3;A;4;4;at_home;teacher;course;m...
1,1;GP;F;17;U;GT3;T;1;1;unknown;other;course;fat...
2,2;GP;F;15;U;LE3;T;1;1;at_home;other;other;moth...
3,3;gp;F;15;;GT3;T;4;2;health;services;home;moth...
4,4;GP;F;16;U;GT3;T;3;3;other;other;home;Father;...


Now try again using `sep`


In [136]:
# your code here
df_mat = pd.read_csv(raw_mat, sep=";")
df_mat.head()

,Unnamed: 0,school,sex,age,address,famsize,pstatus,medu,fedu,mjob,...,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
0,0,GP,F,18,U,GT3,A,4,4,at_home,...,4,3,4,1,1,3,6,5,6,6
1,1,GP,F,17,U,GT3,T,1,1,unknown,...,5,3,3,1,1,3,4,5,5,6
2,2,GP,F,15,U,LE3,T,1,1,at_home,...,4,3,2,2,3,3,10,7,8,10
3,3,gp,F,15,NaN,GT3,T,4,2,health,...,3,2,2,1,1,5,2,15,14,15
4,4,GP,F,16,U,GT3,T,3,3,other,...,4,3,2,1,2,5,4,6,10,10


### Q1.3 Load the Portuguese dataset correctly


In [139]:
# your code here

df_pt = pd.read_csv(raw_pt, sep=";")
df_pt.head()

,Unnamed: 0,school,sex,age,address,famsize,pstatus,medu,fedu,mjob,...,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
0,0,GP,F,18,U,GT3,A,4,4,at_home,...,4,3,4,1,1,3,4,0,11,11
1,1,GP,F,17,u,GT3,T,1,1,at_home,...,5,3,3,1,1,3,2,9,11,11
2,2,GP,female,15,U,LE3,T,1,1,at_home,...,4,3,2,2,3,3,6,12,13,12
3,3,GP,F,15,U,GT3,T,4,2,health,...,3,2,2,1,1,5,0,14,14,14
4,4,?,F,16,U,GT3,T,3,3,other,...,4,3,2,1,2,5,0,11,13,13


**It seems that someone saved the index by mistake... Lets get rid of it!**

### Q1.4 Drop the unnamed column if it exists.
- Use `df.drop(columns=['col_name'], inplace=True)`


In [142]:
df_pt.columns

Index(['Unnamed: 0', 'school', 'sex', 'age', 'address', 'famsize', ' pstatus ',
       ' medu ', ' fedu ', 'mjob ', 'fjob ', 'reason', 'guardian',
       'traveltime', 'studytime', 'failures', 'schoolsup', 'famsup', 'paid',
       'activities', 'nursery', 'higher', 'internet', 'romantic', 'famrel',
       'freetime', 'goout', 'Dalc', 'Walc', 'health', 'absences', 'G1', 'G2',
       'G3'],
      dtype='object')

In [144]:
df_mat.columns

Index(['Unnamed: 0', 'school', 'sex', 'age', 'address', 'famsize', ' pstatus ',
       ' medu ', ' fedu ', 'mjob ', 'fjob ', 'reason', 'guardian',
       'traveltime', 'studytime', 'failures', 'schoolsup', 'famsup', 'paid',
       'activities', 'nursery', 'higher', 'internet', 'romantic', 'famrel',
       'freetime', 'goout', 'Dalc', 'Walc', 'health', 'absences', 'G1', 'G2',
       'G3'],
      dtype='object')

In [146]:
# your code here
df_pt.drop(columns=['Unnamed: 0'], inplace=True)
df_mat.drop(columns=['Unnamed: 0'], inplace=True)

### Q1.5 How many rows and columns are there in each dataset?


In [149]:
# your code here
df_mat.shape
df_pt.shape
print(f"df_mat: {df_mat.shape}, df_pt: {df_pt.shape}")

df_mat: (398, 33), df_pt: (652, 33)


### Q1.6 Check the data type of each column in the Mathematics dataset


In [152]:
# your code here
df_mat.dtypes

school        object
sex           object
age           object
address       object
famsize       object
 pstatus      object
 medu          int64
 fedu          int64
mjob          object
fjob          object
reason        object
guardian      object
traveltime     int64
studytime     object
failures      object
schoolsup     object
famsup        object
paid          object
activities    object
nursery       object
higher        object
internet      object
romantic      object
famrel         int64
freetime       int64
goout          int64
Dalc           int64
Walc           int64
health         int64
absences      object
G1            object
G2            object
G3            object
dtype: object

### Q1.7 Verify descriptive statistics for the Mathematics dataset


In [155]:
# your code here
df_mat.describe()

,medu,fedu,traveltime,famrel,freetime,goout,Dalc,Walc,health
count,398.000000,398.000000,398.000000,398.000000,398.000000,398.000000,398.000000,398.000000,398.000000
mean,2.743719,2.517588,1.447236,3.947236,3.233668,3.108040,1.479899,2.286432,3.550251
std,1.099415,1.091964,0.696143,0.894838,0.995290,1.111373,0.888407,1.286773,1.385872
min,0.000000,0.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
25%,2.000000,2.000000,1.000000,4.000000,3.000000,2.000000,1.000000,1.000000,3.000000
50%,3.000000,2.000000,1.000000,4.000000,3.000000,3.000000,1.000000,2.000000,4.000000
75%,4.000000,3.000000,2.000000,5.000000,4.000000,4.000000,2.000000,3.000000,5.000000
max,4.000000,4.000000,4.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000


## 2. Preprocessing

Now that we have some information about the dataset, it is clear that there are many problems.

Let's preprocess this dataset to make it more viable and clean for future analysis and later insertion into MongoDB.

### Q2.1 Standardize the column names

Suggested steps:
- remove leading and trailing whitespace
- convert names to lowercase
- keep a consistent naming pattern

**Example:**

```python
df.columns = [col.strip().lower() for col in df.columns]
```


In [158]:
# your code here
df_mat.columns = [col.strip().lower() for col in df_mat.columns]
df_pt.columns = [col.strip().lower() for col in df_pt.columns]

### Q2.2 Verify the cleaned column names


In [161]:
# your code here
df_mat.columns


Index(['school', 'sex', 'age', 'address', 'famsize', 'pstatus', 'medu', 'fedu',
       'mjob', 'fjob', 'reason', 'guardian', 'traveltime', 'studytime',
       'failures', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery',
       'higher', 'internet', 'romantic', 'famrel', 'freetime', 'goout', 'dalc',
       'walc', 'health', 'absences', 'g1', 'g2', 'g3'],
      dtype='object')

In [166]:
df_pt.columns


Index(['school', 'sex', 'age', 'address', 'famsize', 'pstatus', 'medu', 'fedu',
       'mjob', 'fjob', 'reason', 'guardian', 'traveltime', 'studytime',
       'failures', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery',
       'higher', 'internet', 'romantic', 'famrel', 'freetime', 'goout', 'dalc',
       'walc', 'health', 'absences', 'g1', 'g2', 'g3'],
      dtype='object')

<details>
<summary style="background-color: #fff3cd; padding: 10px; border-radius: 5px; cursor: pointer;">
    <h2 style="display: inline;">
        Understanding Numerical vs. Categorical Data
        <span style="color: red; font-size: 0.8em;">(click to expand/collapse)</span>
    </h2>
</summary>

It is important to understand **what type of data each column represents**.
Choosing the **correct type** improves **memory usage**, **speeds up operations**, and makes **analysis more reliable**.

---

### 1. Numerical Data

These are values that represent quantities and support mathematical operations.

>**Examples:**
>- age
>- absences
>- G1, G2, G3

### 2. Categorical Data

These represent labels, groups, or categories, not quantities.

>**Examples:**
>- school
>- sex
>- address
>- guardian
>- internet

### Why this distinction matters

If a numeric column is stored as string, mathematical operations may fail.
If a categorical column is inconsistent, grouping and counting become unreliable.

So one of the first steps in Data Wrangling is:

👉 Identify whether each column is numerical or categorical
👉 Fix the dtype accordingly
</details>


### Q2.3 Create a categorical list


In [164]:
# your code here
categorical_cols = ["school", "sex", "address", "famsize", "pstatus", "mjob", "fjob", "reason", "guardian", "schoolsup", "famsup", "paid", "activities", "nursery", "higher", "internet", "romantic"]

In [179]:
df_mat[categorical_cols] = df_mat[categorical_cols].astype("category")
df_pt[categorical_cols] = df_pt[categorical_cols].astype("category")

### Q2.4 Create a numerical list


In [182]:
# your code here
numerical_cols = ["age", "medu", "fedu", "traveltime", "studytime", "failures", "famrel", "freetime", "goout", "dalc", "walc", "health", "absences", "g1", "g2", "g3"]

### Q2.5 Inspect unique values in selected categorical columns


In [185]:
# your code here
for col in categorical_cols:
    print(f"\nColumn: {col}")
    print(df_mat[col].unique())




Column: school
['GP', 'gp', ' Gabriel Pereira', ' ?', 'GP ', NaN, 'unknown', 'MS']
Categories (7, object): [' ?', ' Gabriel Pereira', 'GP', 'GP ', 'MS', 'gp', 'unknown']

Column: sex
['F', 'M', 'm', 'F ', 'f', 'female', 'male', 'M ']
Categories (8, object): ['F', 'F ', 'M', 'M ', 'f', 'female', 'm', 'male']

Column: address
['U', NaN, 'R', 'urban', 'r', 'u', ' ?']
Categories (6, object): [' ?', 'R', 'U', 'r', 'u', 'urban']

Column: internet
['no', 'yes', 'Yes', 'Y', 'N', ' ?', 'yes ', 'unknown', NaN]
Categories (8, object): [' ?', 'N', 'Y', 'Yes', 'no', 'unknown', 'yes', 'yes ']

Column: higher
['yes', 'Yes', 'unknown', 'no', 'YES', ' y', NaN]
Categories (6, object): [' y', 'YES', 'Yes', 'no', 'unknown', 'yes']


In [188]:
# your code here
for col in categorical_cols:
    print(f"\nColumn: {col}")
    print(df_pt[col].unique())


Column: school
['GP', 'GP ', ' ?', 'unknown', ' Gabriel Pereira', ..., NaN, 'MS', 'ms', 'Mousinho da Silveira', 'MS ']
Length: 11
Categories (10, object): [' ?', ' Gabriel Pereira', 'GP', 'GP ', ..., 'Mousinho da Silveira', 'gp', 'ms', 'unknown']

Column: sex
['F', 'female', 'M', 'F ', 'male', 'M ', 'm', 'f']
Categories (8, object): ['F', 'F ', 'M', 'M ', 'f', 'female', 'm', 'male']

Column: address
['U', 'u', 'urban', 'R', ' ?', ..., 'R ', 'U ', 'unknown', 'rural', 'r']
Length: 11
Categories (10, object): [' ?', 'R', 'R ', 'U', ..., 'rural', 'u', 'unknown', 'urban']

Column: internet
['no', 'yes', 'Y', NaN, 'Yes', 'yes ', ' ?', 'N', 'No', 'no ']
Categories (9, object): [' ?', 'N', 'No', 'Y', ..., 'no', 'no ', 'yes', 'yes ']

Column: higher
['yes', 'Yes', ' y', 'YES', 'no', NaN, ' ?', 'NO', 'unknown', ' n']
Categories (9, object): [' ?', ' n', ' y', 'NO', ..., 'Yes', 'no', 'unknown', 'yes']


### Q2.6 Replace inconsistent missing markers with `NaN`

>**Examples of markers that may appear:**
>- `?`
>- ` ?`
>- `unknown`
>- `N/A`
>- empty strings
---
>**Example:**
>
>```python
>df.replace({"?": np.nan, "unknown": np.nan}, inplace=True)
>```


In [190]:
# your code here
df_mat.replace(
    {
        "?": np.nan,
        " ?": np.nan,
        "unknown": np.nan,
        "N/A": np.nan,
        "": np.nan
    },
    inplace=True
)

df_pt.replace(
    {
        "?": np.nan,
        " ?": np.nan,
        "unknown": np.nan,
        "N/A": np.nan,
        "": np.nan
    },
    inplace=True
)

C:\Users\ineko\AppData\Local\Temp\ipykernel_8232\788112765.py:2: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  df_mat.replace(
C:\Users\ineko\AppData\Local\Temp\ipykernel_8232\788112765.py:13: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  df_pt.replace(


### Q2.7 Check for missing values in every column


In [192]:
# your code here
df_mat.isna().sum()

school        11
sex            0
age            0
address       11
famsize        0
pstatus        0
medu           0
fedu           0
mjob          12
fjob          11
reason        11
guardian      11
traveltime     0
studytime      0
failures       0
schoolsup      0
famsup         0
paid           0
activities     0
nursery        0
higher        11
internet      11
romantic       0
famrel         0
freetime       0
goout          0
dalc           0
walc           0
health         0
absences       0
g1             0
g2             0
g3             0
dtype: int64

In [194]:
df_pt.isna().sum()

school        19
sex            0
age            0
address       19
famsize        0
pstatus        0
medu           0
fedu           0
mjob          19
fjob          19
reason        19
guardian      19
traveltime     0
studytime      0
failures       0
schoolsup      0
famsup         0
paid           0
activities     0
nursery        0
higher        19
internet      19
romantic       0
famrel         0
freetime       0
goout          0
dalc           0
walc           0
health         0
absences       0
g1             0
g2             0
g3             0
dtype: int64

### Q2.8 Which columns are the most problematic in terms of missing values?


In [ ]:
# your code here
#As colunas mais problemáticas são variáveis categóricas 
#(mjob, fjob, reason, guardian, school, address, higher, internet), pois concentram praticamente todos os valores em falta.

### Q2.9 Define a function that checks the percentage of missing values for a given column

>**Example:**
>
>```python
>def percentage_missing_values(df: pd.DataFrame, col: str) -> float:
>    # your code here
>    return percentage
>```


In [202]:
# your code here
def percentage_missing_values(df: pd.DataFrame, col: str) -> float:
    return (df[col].isna().sum() / len(df)) * 100

percentage_missing_values(df_mat, "guardian")

2.763819095477387

## 3. Standardizing categorical values

Now that missing markers are fixed, we must standardize inconsistent categories.

>**Examples of dirty values that may appear:**
>- `male`, `M`, `m`, `M `
>- `female`, `F`, `f`, `F `
>- `urban`, `U`, `u`
>- `rural`, `R`, `r`
>- `yes`, `Yes`, `Y`
>- `no`, `No`, `N`


In [205]:
 pd.Series

pandas.core.series.Series

In [207]:
s = pd.Series([10, 20, 30])
print(s)

0    10
1    20
2    30
dtype: int64


### Q3.1 Define helper functions to standardize selected categorical fields


In [209]:
# your code here
def clean_category(series: pd.Series) -> pd.Series:
    return (
        series
        .astype(str)
        .str.strip()
        .str.lower()
    )

### Q3.2 Apply the categorical standardization to both DataFrames


In [211]:
# your code here
for col in categorical_cols:
    df_mat[col] = clean_category(df_mat[col])
    df_pt[col] = clean_category(df_pt[col])

### Q3.3 Recheck the unique values after standardization


In [213]:
# your code here
for col in categorical_cols:
    print(f"\nColumn: {col}")
    print(df_mat[col].unique())


Column: school
['gp' 'gabriel pereira' 'nan' 'ms']

Column: sex
['f' 'm' 'female' 'male']

Column: address
['u' 'nan' 'r' 'urban']

Column: internet
['no' 'yes' 'y' 'n' 'nan']

Column: higher
['yes' 'nan' 'no' 'y']


In [215]:
# your code here
for col in categorical_cols:
    print(f"\nColumn: {col}")
    print(df_pt[col].unique())


Column: school
['gp' 'nan' 'gabriel pereira' 'ms' 'mousinho da silveira']

Column: sex
['f' 'female' 'm' 'male']

Column: address
['u' 'urban' 'r' 'nan' 'rural']

Column: internet
['no' 'yes' 'y' 'nan' 'n']

Column: higher
['yes' 'y' 'no' 'nan' 'n']


## 4. Fixing column types

Some numerical columns seems polluted with strings.

>**Examples:**
>- `14 points`
>- `2h`
>- ` 17 `
>- `15.0`

### Q4.1 Convert polluted numeric columns into proper numeric dtype


In [217]:
df_mat[numerical_cols] = df_mat[numerical_cols].replace(r"[^\d.]", "", regex=True)
df_pt[numerical_cols] = df_pt[numerical_cols].replace(r"[^\d.]", "", regex=True)

In [219]:
df_mat[numerical_cols] = df_mat[numerical_cols].apply(pd.to_numeric)
df_pt[numerical_cols] = df_pt[numerical_cols].apply(pd.to_numeric)

### Q4.2 Check `.info()` again after type conversion


In [221]:
# your code here
df_mat.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 398 entries, 0 to 397
Data columns (total 33 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   school      398 non-null    object 
 1   sex         398 non-null    object 
 2   age         398 non-null    float64
 3   address     398 non-null    object 
 4   famsize     398 non-null    object 
 5   pstatus     398 non-null    object 
 6   medu        398 non-null    int64  
 7   fedu        398 non-null    int64  
 8   mjob        386 non-null    object 
 9   fjob        387 non-null    object 
 10  reason      387 non-null    object 
 11  guardian    387 non-null    object 
 12  traveltime  398 non-null    int64  
 13  studytime   398 non-null    float64
 14  failures    398 non-null    float64
 15  schoolsup   398 non-null    object 
 16  famsup      398 non-null    object 
 17  paid        398 non-null    object 
 18  activities  398 non-null    object 
 19  nursery     398 non-null    o

## 5. Missing value imputation

Now that types are cleaner, we can impute missing values more safely.

### Q5.1 Impute missing values for numerical columns
- A reasonable first option is the median


In [223]:
# your code here
for col in numerical_cols:
    df_mat[col].fillna(df_mat[col].median(), inplace=True)
    df_pt[col].fillna(df_pt[col].median(), inplace=True)

C:\Users\ineko\AppData\Local\Temp\ipykernel_8232\3457430816.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_mat[col].fillna(df_mat[col].median(), inplace=True)
C:\Users\ineko\AppData\Local\Temp\ipykernel_8232\3457430816.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For

### Q5.2 Impute missing values for categorical columns
- A reasonable first option is the mode
- If no mode is available, use a fixed value such as `unknown`


In [231]:
# your code here
for col in categorical_cols:
    mode_value = df_mat[col].mode()

    if len(mode_value) > 0:
        df_mat[col].fillna(mode_value[0], inplace=True)
    else:
        df_mat[col].fillna("unknown", inplace=True)


for col in categorical_cols:
    mode_value = df_pt[col].mode()

    if len(mode_value) > 0:
        df_pt[col].fillna(mode_value[0], inplace=True)
    else:
        df_pt[col].fillna("unknown", inplace=True)

C:\Users\ineko\AppData\Local\Temp\ipykernel_8232\2216170639.py:6: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_mat[col].fillna(mode_value[0], inplace=True)
C:\Users\ineko\AppData\Local\Temp\ipykernel_8232\2216170639.py:15: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For examp

### Q5.3 Verify whether missing values still exist


In [236]:
# your code here
df_mat.isna().sum()[df_mat.isna().sum() > 0]

mjob        12
fjob        11
reason      11
guardian    11
dtype: int64

In [238]:
cols_missing = ["mjob", "fjob", "reason", "guardian"]

for col in cols_missing:
    df_mat[col].fillna(df_mat[col].mode()[0], inplace=True)
    df_pt[col].fillna(df_pt[col].mode()[0], inplace=True)

C:\Users\ineko\AppData\Local\Temp\ipykernel_8232\1203140900.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_mat[col].fillna(df_mat[col].mode()[0], inplace=True)
C:\Users\ineko\AppData\Local\Temp\ipykernel_8232\1203140900.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

Fo

In [242]:
df_pt.isna().sum()[df_mat.isna().sum() > 0]

Series([], dtype: int64)

In [244]:
df_mat.isna().sum()[df_mat.isna().sum() > 0]

Series([], dtype: int64)

## 6. Duplicates and outliers

### Q6.1 Check duplicated rows


In [246]:
# your code here
df_mat.duplicated().sum()

3

In [248]:
df_pt.duplicated().sum()

3

### Q6.2 Remove duplicated rows if necessary


In [250]:
# your code here
df_mat.drop_duplicates(inplace=True)
df_pt.drop_duplicates(inplace=True)

### Q6.3 Verify if there are **outliers** in numerical columns using the `IQR method`

To detect outliers in a numerical column, we can use the **Interquartile Range (IQR) method**.

The formula works as follows:

- Compute the 1st quartile (Q1)
- Compute the 3rd quartile (Q3)
- Compute the **IQR**

Outliers are any observations outside:

$$
\text{{Lower Bound}} = Q1 - 1.5 \times \text{{IQR}}
$$

$$
\text{{Upper Bound}} = Q3 + 1.5 \times \text{{IQR}}
$$

Define a function that verifies whether a column contains outliers:

```python
def verify_outliers(df: pd.DataFrame, col: str) -> bool:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    # continue from here
```


In [253]:
def verify_outliers(df: pd.DataFrame, col: str) -> bool:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    return ((df[col] < lower_bound) | (df[col] > upper_bound)).any()

### Q6.4 Apply the outlier check to the numerical columns


In [255]:
# your code here
for col in numerical_cols:
    print(f"{col}: {verify_outliers(df_mat, col)}")

age: True
medu: False
fedu: True
traveltime: True
studytime: True
failures: True
famrel: True
freetime: True
goout: False
dalc: True
walc: False
health: False
absences: True
g1: True
g2: True
g3: False


## 7. Preparing the dataset for Ingestion Pipelines

At the end of this notebook, the data should be clean enough to be loaded later into MongoDB.

### Q7.1 Add a `source_file` column to each DataFrame

Expected values:
- `student-mat-dirty.csv`
- `student-por-dirty.csv`


In [257]:
# your code here
df_mat["source_file"] = "student-mat-dirty.csv"
df_pt["source_file"] = "student-por-dirty.csv"

### Q7.2 Concatenate both cleaned DataFrames into one unified dataset

>**Example:**
>
>```python
>df_combined = pd.concat([df1, df2], ignore_index=True)
>```


In [259]:
# your code here
df_combined = pd.concat([df_mat, df_pt], ignore_index=True)

### Q7.3 Inspect the final structure of the cleaned dataset


In [263]:
# your code here
df_combined.shape


(1044, 34)

### Q7.4 Inspect a small sample of the final cleaned dataset

>**Example:**
>
>```python
>df_combined.sample(5)
>```


In [267]:
# your code here
df_combined.sample(5)

,school,sex,age,address,famsize,pstatus,medu,fedu,mjob,fjob,...,freetime,goout,dalc,walc,health,absences,g1,g2,g3,source_file
127,gp,f,19.0,u,GT3,T,0,1,at_home,other,...,4,2,1,1,5,2.0,7.0,8.0,9.0,student-mat-dirty.csv
446,gp,f,15.0,urban,LE3,T,4,2,health,other,...,3,3,1,1,5,0.0,16.0,14.0,16.0,student-por-dirty.csv
792,gp,f,17.0,u,GT3,A,2,2,at_home,at_home,...,3,1,1,2,4,18.0,10.0,12.0,14.0,student-por-dirty.csv
904,ms,f,16.0,r,LE3,T,1,1,at_home,other,...,3,2,1,1,1,0.0,16.0,17.0,18.0,student-por-dirty.csv
97,gp,female,16.0,u,GT3,T,2,1,other,other,...,3,5,1,1,5,2.0,8.0,9.0,10.0,student-mat-dirty.csv


### Q7.5 Export the cleaned dataset to CSV

Suggested file:
- `./student-performance-cleaned.csv`


In [269]:
# your code here
df_combined.to_csv("./student-performance-cleaned.csv", index=False)

## Final reflection

After finishing this notebook, you should be able to explain:
- what was dirty in the dataset
- which steps were needed to clean it
- why this preprocessing should happen **before** inserting the data in a database
- how the cleaned dataset is now more suitable for later EDA and database loading
